# Mind-Virus Simulation
Runs the `mindvirus` simulation and analyzes results inline.
Runtime: any for API-backed runs; GPU (T4+) for HF-backed runs.

In [ ]:
# Install from GitHub (or from a Drive clone: %pip install -e /content/drive/MyDrive/multiagent)
%pip install -q "mindvirus[hf] @ git+https://github.com/stevenokada/multiagent"

In [ ]:
import os
try:
    from google.colab import userdata, drive
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # for ValuePrism / gated checkpoints
    except Exception:
        pass
    drive.mount("/content/drive")
    RUNS_DIR = "/content/drive/MyDrive/mindvirus-runs"
except ImportError:  # not on Colab
    RUNS_DIR = "runs"
os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
from mindvirus.config import Config, ModelConfig, CaptureConfig

cfg = Config(
    agent_model=ModelConfig(backend="anthropic", model="claude-haiku-4-5"),
    judge_model=ModelConfig(backend="anthropic", model="claude-haiku-4-5"),
    # For a local model instead:
    # agent_model=ModelConfig(backend="hf", model="Qwen/Qwen2.5-7B-Instruct"),
    # capture=CaptureConfig(enabled=True, layers="all", positions="last", calls=["agent_turn"]),
    n_agents=10, rounds=15, probe_every=5, seed=0,
    payload_id="honesty-absolutism", n_patient_zero=1,
    battery_source="hand",   # "valueprism" needs HF_TOKEN + accepted dataset terms
    runs_dir=RUNS_DIR,
)

In [ ]:
from mindvirus import run_experiment
run_dir = run_experiment(cfg)
run_dir

In [ ]:
from mindvirus import load_run, summarize, plots
run = load_run(run_dir)
summarize(run)

In [ ]:
plots.infection_curve(run)
plots.probe_trajectories(run)

## Control run
Re-run with `n_patient_zero=0` (same seed) and compare `summarize` outputs / trajectories.

## Interp later
Every model call is in `calls.jsonl` (with `activation_path` when capture was on).
Load tensors with `torch.load(path)` and join to calls/judgements by `call_id`.